## Example 1: Write and read a .laz file through laspy's lazrs backend

In [ ]:
import os
import tempfile

import laspy
import numpy as np

# Example 1: write a point cloud to .laz through the lazrs backend and read it back unchanged
rng = np.random.default_rng(1)
header = laspy.LasHeader(point_format=3, version="1.2")
header.scales = [0.01, 0.01, 0.01]
cloud = laspy.LasData(header)
cloud.x, cloud.y, cloud.z = rng.uniform(0, 500, (3, 50_000))
cloud.intensity = rng.integers(0, 65_535, 50_000, dtype=np.uint16)
with tempfile.TemporaryDirectory() as tmp_dir:
    path = os.path.join(tmp_dir, "cloud.laz")
    cloud.write(path, laz_backend=laspy.LazBackend.Lazrs)
    back = laspy.read(path, laz_backend=laspy.LazBackend.Lazrs)
    assert back.header.are_points_compressed, "the file was not written compressed"
    for field in ("X", "Y", "Z", "intensity"):
        assert np.array_equal(np.asarray(cloud[field]), np.asarray(back[field])), field
    print(f"{len(back.points)} points round-tripped through {os.path.getsize(path)} compressed bytes")

## Example 2: The multithreaded backend agrees with the single-threaded one

In [ ]:
import io

import laspy
import numpy as np

# Example 2: the multithreaded backend (LazrsParallel) writes the same points as the single-threaded one
rng = np.random.default_rng(2)
header = laspy.LasHeader(point_format=1, version="1.2")
cloud = laspy.LasData(header)
cloud.x, cloud.y, cloud.z = rng.uniform(0, 100, (3, 120_000))
cloud.gps_time = np.sort(rng.uniform(0, 1e5, 120_000))
decoded = {}
for backend in (laspy.LazBackend.Lazrs, laspy.LazBackend.LazrsParallel):
    buffer = io.BytesIO()
    cloud.write(buffer, do_compress=True, laz_backend=backend)
    buffer.seek(0)
    decoded[backend.name] = laspy.read(buffer, laz_backend=backend)
a, b = decoded["Lazrs"], decoded["LazrsParallel"]
assert np.array_equal(np.asarray(a.X), np.asarray(b.X)) and np.array_equal(np.asarray(a.gps_time), np.asarray(b.gps_time))
print("single-threaded and parallel backends agree on", len(a.points), "points")

## Example 3: Stream a compressed file in chunks

In [ ]:
import io

import laspy
import numpy as np

# Example 3: stream a compressed file in chunks, so a large cloud never has to fit in memory at once
rng = np.random.default_rng(3)
cloud = laspy.LasData(laspy.LasHeader(point_format=3, version="1.4"))
cloud.x, cloud.y, cloud.z = rng.uniform(0, 1_000, (3, 200_000))
cloud.classification = rng.integers(1, 7, 200_000, dtype=np.uint8)
buffer = io.BytesIO()
cloud.write(buffer, do_compress=True, laz_backend=laspy.LazBackend.Lazrs)
buffer.seek(0)
counts = np.zeros(256, dtype=np.int64)
with laspy.open(buffer, laz_backend=laspy.LazBackend.Lazrs) as reader:
    for chunk in reader.chunk_iterator(30_000):
        counts += np.bincount(np.asarray(chunk.classification), minlength=256)
assert np.array_equal(counts, np.bincount(np.asarray(cloud.classification), minlength=256))
print("per-class point counts from streamed chunks:", {c: int(n) for c, n in enumerate(counts) if n})

## Example 4: Append points to an existing .laz file

In [ ]:
import os
import tempfile

import laspy
import numpy as np

# Example 4: append points to an existing .laz file (laspy can append to LAZ only through lazrs)
rng = np.random.default_rng(4)
first = laspy.LasData(laspy.LasHeader(point_format=3, version="1.2"))
first.x, first.y, first.z = rng.uniform(0, 10, (3, 10_000))
more = laspy.ScaleAwarePointRecord.zeros(5_000, header=first.header)
more.x, more.y, more.z = rng.uniform(0, 10, (3, 5_000))
with tempfile.TemporaryDirectory() as tmp_dir:
    path = os.path.join(tmp_dir, "tiles.laz")
    first.write(path, laz_backend=laspy.LazBackend.Lazrs)
    with laspy.open(path, mode="a", laz_backend=laspy.LazBackend.Lazrs) as appender:
        appender.append_points(more)
    merged = laspy.read(path)
    assert len(merged.points) == 15_000 and merged.header.point_count == 15_000
    assert np.array_equal(np.asarray(merged.X[10_000:]), np.asarray(more.X))
print("appended 5,000 points to a 10,000-point .laz file:", len(merged.points))

## Example 5: lazrs's own compress and decompress functions

In [ ]:
import laspy
import lazrs
import numpy as np

# Example 5: lazrs's own API: compress raw LAS point records (one thread and many) and decompress them
# (decompress_points is called single-threaded: in laz 0.12.2 its parallel path cuts the input 8 bytes short and
# panics "mid > len", an upstream defect laspy never reaches; laspy's parallel reader is shown in Example 2)
rng = np.random.default_rng(5)
cloud = laspy.LasData(laspy.LasHeader(point_format=1, version="1.2"))
cloud.x, cloud.y, cloud.z = rng.uniform(0, 50, (3, 120_000))
cloud.gps_time = np.sort(rng.uniform(0, 1e4, 120_000))
raw = np.frombuffer(cloud.points.array.tobytes(), dtype=np.uint8)
vlr = lazrs.LazVlr.new_for_compression(1, 0)
for parallel in (False, True):
    compressed = bytes(lazrs.compress_points(vlr, raw, parallel))
    out = np.zeros_like(raw)
    lazrs.decompress_points(compressed, vlr.record_data(), out, False)
    assert np.array_equal(out, raw), f"round trip differs (parallel compression={parallel})"
    print(f"parallel compression={parallel}: {raw.nbytes} raw bytes -> {len(compressed)} compressed, round trip exact")